# Runtime curve walkthrough: flood × real estate

Reads the proposed model-v0.2 artifact and calls the curves through the cell's reference evaluator, the way a
Hazard M3 consumer would. Damage only: no frequency, no yearly loss. See
[`00_curve_curation_walkthrough.ipynb`](00_curve_curation_walkthrough.ipynb) for how the curves are built.

In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "docs/cells").is_dir() and (p / "scripts/reference_helpers").is_dir():
            return p
    raise RuntimeError("Run this notebook from inside the Damage_Modeling repository.")


ROOT = find_repo_root()
CELL_ID = "flood_real_estate"
PROP = ROOT / "docs/cells" / CELL_ID / "proposed"
ARTIFACT_PATH = PROP / "flood_real_estate__model_v0_2__docs_r1__curve_artifact.json"
REG = "flood_real_estate__model_v0_1__docs_r1"  # tag of the cell registers (source, claims, tiers, decisions)
OUT = ROOT / "notebooks/flood/real_estate/outputs"
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "scripts/reference_helpers"))
import flood_real_estate_curve_eval as ev  # the cell's reference evaluator

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

ART = json.loads(ARTIFACT_PATH.read_text())
# Replacement cost new from the Hazard reference profiles r1 (Q3 2025 Dallas prices). Illustration only:
# the artifacts withhold scenario dollars until the consumer qualifies each building value.
BUDGETS_USD = {"RE_RES_SFD_1S_SLAB": 405_000, "RE_IND_WAREHOUSE": 11_000_000, "RE_COM_OFFICE_3S_SLAB": 10_800_000}
NAMES = {"RE_RES_SFD_1S_SLAB": "House", "RE_IND_WAREHOUSE": "Warehouse", "RE_COM_OFFICE_3S_SLAB": "Office",
         "RE_COM_LOWRISE": "Retail (unbound)"}


def table(name: str) -> pd.DataFrame:
    return pd.read_csv(PROP / name)


def save(fig, name: str) -> None:
    fig.savefig(OUT / name, dpi=110, bbox_inches="tight", metadata={"Software": None})


def decision_log(path: Path) -> pd.DataFrame:
    """Decision-log rows: ID, whether JL decided it, and the decision text (shortened)."""
    rows = []
    for line in path.read_text().splitlines():
        if not line.startswith("| D-"):
            continue
        cells = [c.strip() for c in line.strip().strip("|").split(" | ")]
        text = re.sub(r"\*\*", "", cells[1])
        rows.append({"id": cells[0], "status": "JL decided" if "JL decided" in cells[1] else "proposed",
                      "decision": text if len(text) <= 240 else text[:240] + " …"})
    return pd.DataFrame(rows)


print("artifact:", ARTIFACT_PATH.relative_to(ROOT))
print(ART["cell_id"], "|", ART["semantic_damage_model_version"], "|", ART["damage_code_id"],
      "| promotion:", ART["promotion_status"], "| canonical:", ART["canonical_runtime_artifact"])


def call(request: dict) -> dict:
    """Send one request through the cell's reference evaluator; return a flat row."""
    try:
        out = ev.evaluate_damage_call(ART, request)
    except ev.DamageCallError as err:
        return {"status": "error", "curve_id": None, "damage_ratio": None, "reason": err.code}
    r = out["failure_unit_results"][0]
    return {"status": r["status"], "curve_id": r["curve_id"], "damage_ratio": r["scalar_central_dr"],
            "reason": "; ".join(r["withheld_reason_codes"]) or None}

artifact: docs/cells/flood_real_estate/proposed/flood_real_estate__model_v0_2__docs_r1__curve_artifact.json
flood_real_estate | model v0.2 | FLOOD_REAL_ESTATE_HAZUS_NACCS_SCREENING_V0_2 | promotion: proposed | canonical: False


## 1. What the artifact contains

In [2]:
identity = pd.DataFrame([
    {"field": "cell", "value": ART["cell_id"]},
    {"field": "damage code", "value": ART["damage_code_id"]},
    {"field": "model / docs", "value": f'{ART["semantic_damage_model_version"]} / {ART["documentation_revision"]}'},
    {"field": "lifecycle / promotion", "value": f'{ART["lifecycle_state"]} / {ART["promotion_status"]}'},
    {"field": "canonical runtime artifact", "value": ART["canonical_runtime_artifact"]},
    {"field": "model grade", "value": ART["model_grade"]},
])
display(identity)
units = pd.DataFrame([{k: u.get(k) for k in ("id", "treatment", "denominator")} for u in ART["failure_units"]])
display(units)
pathways = pd.DataFrame([{
    "pathway": p["pathway_id"],
    "axis field": p["hazard_axis"]["preferred_input_field"],
    "unit": p["hazard_axis"]["unit"],
    "valid range": p["hazard_axis"]["valid_range"],
    "selectors": ", ".join(s["field"] for s in p["selector_logic"] if s.get("required")),
    "conditioners": ", ".join(c["field"] for c in p["conditioner_logic"]) or "-",
    "records": len(p["curve_records"]),
} for p in ART["pathways"]])
display(pathways)

,field,value
0,cell,flood_real_estate
1,damage code,FLOOD_REAL_ESTATE_HAZUS_NACCS_SCREENING_V0_2
2,model / docs,model v0.2 / docs r1
3,lifecycle / promotion,scaffold / proposed
4,canonical runtime artifact,False
5,model grade,screening_source_native_adopted_tables


,id,treatment,denominator
0,FRE_BUILDING_STRUCTURE,primary_nonzero,"structure replacement cost new of the same reference building (no land, contents, site works or depreciation)"
1,FRE_BUILDING_CONTENTS,withheld,contents value (deferred)


,pathway,axis field,unit,valid range,selectors,conditioners,records
0,flood_inundation_contact,flood_depth_above_first_floor_ft,ft,"[-4.0, 24.0]",reference_building_id,flood_water_type,7


## 2. Request contract

Two ways to send depth: above ground (preferred; the evaluator subtracts the floor height) or already above the first floor.

In [3]:
display(pd.DataFrame(ART["evaluation_contract"]["accepted_axis_payloads"]))
display(pd.DataFrame(ART["pathways"][0]["conditioner_logic"])[["field", "required", "allowed", "withhold_otherwise"]])
print(ART["pathways"][0]["hazard_axis"]["bridge"])

,mode,required_fields,optional_fields
0,depth_above_grade_ft,[flood_depth_above_grade_ft],[first_floor_height_ft]
1,depth_above_first_floor_ft,[flood_depth_above_first_floor_ft],NaN


,field,required,allowed,withhold_otherwise
0,flood_water_type,True,"[freshwater_riverine_or_pluvial, coastal_surge_saline, unknown]",FLOOD_WATER_TYPE_UNKNOWN


flood_depth_above_first_floor_ft = flood_depth_above_grade_ft - first_floor_height_ft; depth above grade <= 0 is not inundated (DR 0); a consumer sending floor-relative depth applies the dry rule itself


## 3. Calling the curves

Depth above ground, each building at its reference floor height, river water.

In [4]:
rows = []
for b in ("RE_RES_SFD_1S_SLAB", "RE_IND_WAREHOUSE", "RE_COM_OFFICE_3S_SLAB"):
    for depth in (0.0, 0.5, 1, 2, 3, 4, 6, 8, 12):
        res = call({"pathway_id": "flood_inundation_contact", "reference_building_id": b,
                    "flood_water_type": "freshwater_riverine_or_pluvial", "flood_depth_above_grade_ft": float(depth)})
        rows.append({"building": NAMES[b], "depth above ground (ft)": depth, "damage_ratio": res["damage_ratio"]})
grid = pd.DataFrame(rows).pivot(index="building", columns="depth above ground (ft)", values="damage_ratio").rename(columns=lambda c: f"{c:g} ft")
display(grid.style.format("{:.1%}"))

depth above ground (ft),0 ft,0.5 ft,1 ft,2 ft,3 ft,4 ft,6 ft,8 ft,12 ft
building,,,,,,,,,
House,0.0%,10.5%,15.5%,25.2%,34.0%,41.8%,54.5%,64.0%,75.5%
Office,0.0%,4.2%,8.8%,14.8%,20.5%,26.5%,37.2%,43.2%,56.2%
Warehouse,0.0%,1.2%,3.8%,7.3%,10.2%,12.5%,18.2%,24.2%,40.0%


Coastal surge on the retail reference returns the NACCS curve with its min/max building-variant envelope (not a probability spread):

In [5]:
out = ev.evaluate_damage_call(ART, {"pathway_id": "flood_inundation_contact", "reference_building_id": "RE_COM_LOWRISE",
                                  "flood_water_type": "coastal_surge_saline", "flood_depth_above_grade_ft": 3.0})
r = out["failure_unit_results"][0]
print(r["curve_id"], "central", r["scalar_central_dr"], "| envelope", r.get("scenario_drs"))

FRE_COM_COAST_NACCS_P2 central 0.33749999999999997 | envelope {'naccs_min_building_variant': 0.255, 'naccs_max_building_variant': 0.4125}


## 4. Fail-closed behaviour

In [6]:
base = {"pathway_id": "flood_inundation_contact", "reference_building_id": "RE_RES_SFD_1S_SLAB",
        "flood_water_type": "freshwater_riverine_or_pluvial", "flood_depth_above_grade_ft": 3.0}
cases = {
    "valid request (3 ft above ground)": base,
    "dry ground (0 ft above ground)": {**base, "flood_depth_above_grade_ft": 0.0},
    "floor-relative depth below the floor (-0.25 ft)": {k: v for k, v in base.items() if k != "flood_depth_above_grade_ft"} | {"flood_depth_above_first_floor_ft": -0.25},
    "above the published range (30 ft)": {**base, "flood_depth_above_grade_ft": 30.0},
    "unknown water type": {**base, "flood_water_type": "unknown"},
    "both depth modes at once": {**base, "flood_depth_above_first_floor_ft": 2.0},
    "contents": {**base, "failure_unit_id": "FRE_BUILDING_CONTENTS"},
}
display(pd.DataFrame([{"case": k, **call(v)} for k, v in cases.items()]))
display(Markdown("The floor-relative case shows why the consumer must apply the dry-ground rule itself when it sends "
                 "depth above the floor: the published curve is not zero just below the floor."))

,case,status,curve_id,damage_ratio,reason
0,valid request (3 ft above ground),conditional,FRE_RES_HAZUS_DF129,0.340,NaN
1,dry ground (0 ft above ground),conditional,FRE_RES_HAZUS_DF129,0.000,NaN
2,floor-relative depth below the floor (-0.25 ft),conditional,FRE_RES_HAZUS_DF129,0.105,NaN
3,above the published range (30 ft),withheld,FRE_RES_HAZUS_DF129,NaN,ABOVE_SOURCE_VALID_RANGE
4,unknown water type,withheld,NaN,NaN,FLOOD_WATER_TYPE_UNKNOWN
5,both depth modes at once,error,NaN,NaN,AXIS_PAYLOAD_AMBIGUOUS
6,contents,withheld,NaN,NaN,NO_RUNTIME_CURVE_FOR_PATHWAY_UNIT; CONTENTS_DEFERRED_SEPARATE_FAILURE_UNIT


The floor-relative case shows why the consumer must apply the dry-ground rule itself when it sends depth above the floor: the published curve is not zero just below the floor.

## 5. From damage ratio to dollars (illustration)

The Hazard profile budgets are used only to show `loss = DR × replacement cost new`; the artifact withholds dollars until Hazard qualifies each value.

In [7]:
rows = []
for b, budget in BUDGETS_USD.items():
    res = call({"pathway_id": "flood_inundation_contact", "reference_building_id": b,
                "flood_water_type": "freshwater_riverine_or_pluvial", "flood_depth_above_grade_ft": 3.0})
    rows.append({"building": NAMES[b], "replacement cost new": budget, "DR at 3 ft above ground": res["damage_ratio"],
                 "loss (illustration)": res["damage_ratio"] * budget})
display(pd.DataFrame(rows).style.format({"replacement cost new": "${:,.0f}", "DR at 3 ft above ground": "{:.1%}",
                                         "loss (illustration)": "${:,.0f}"}))

,building,replacement cost new,DR at 3 ft above ground,loss (illustration)
0,House,"$405,000",34.0%,"$137,700"
1,Warehouse,"$11,000,000",10.2%,"$1,127,500"
2,Office,"$10,800,000",20.5%,"$2,214,000"


## 6. Capability

In [8]:
cap = ART["capability_declaration"]
rows = []
for pc in cap["pathway_capabilities"]:
    rows.append({"pathway": pc["pathway_id"], "failure-unit DR": pc["failure_unit_scalar_dr"],
                 "scenario loss": pc["scenario_loss_given_value_basis"], "spread": pc["curve_intrinsic_spread"],
                 "withheld units": ", ".join(w["failure_unit_id"] for w in pc.get("withheld_failure_units", []))})
display(pd.DataFrame(rows))
print("Annual and tail metrics:", cap["consumer_annual_metrics"]["status_before_promotion"],
      "(owner:", cap["consumer_annual_metrics"]["computation_owner"] + ")")
print("Promotion gate:", cap["promotion_gate"]["status"])
for item in cap["promotion_gate"]["required_before_canonical_use"]:
    print("  -", item)

,pathway,failure-unit DR,scenario loss,spread,withheld units
0,flood_inundation_contact,conditional,withheld,nonprobabilistic_epistemic_envelope,FRE_BUILDING_CONTENTS


Annual and tail metrics: withheld_noncanonical_proposal (owner: downstream_consumer)
Promotion gate: blocked
  - the reviewer accepts scope D-FRE-01 to D-FRE-06 and selections D-FRE-16 to D-FRE-28
  - consumer value qualification of each reference budget against the structure denominator
  - consumer integration resolves record-level ranges, routing and the dry-ground rule
  - independent review of the package


## 7. Not in this notebook

- Frequency and yearly loss belong to Hazard.
- The NFIP claims check is `scripts/reference_helpers/flood_real_estate_nfip_claims_check.py` (it downloads claims
  from OpenFEMA; results in the cell's `NFIP_CLAIMS_*` tables).